<b>Winner Determination for Combinatorial Auction</b>

In [1]:
# data
Slots = range(1,7)
Price = {1 : 100, 2 : 110, 3 : 75, 4 : 110, 5 : 95, 6 : 90, 7 : 65, 8 : 75}
Airline = {1 : 'OptAir', 2 : 'OptAir', 3 : 'OptAir', 4 : 'Gamma', 5 : 'Gamma', 6 : 'Gamma', 7 : 'Atlantis', 8 : 'Atlantis' }
SlotsInBid = {1 : {1, 2, 4}, 2 : {1, 2, 5}, 3 : {3, 6}, 4 : {2, 3, 6}, 5 : {1, 4, 5}, 6 : {1, 2, 5}, 7 : {2, 3}, 8 : {1, 5}}

In [2]:
# create set of Bids and set of Airlines
Bids = Price.keys()
Airlines = set(Airline.values())

In [3]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [4]:
# variables
select = mdl.binary_var_dict(Bids, name = "select")

In [5]:
# objective
mdl.maximize(mdl.sum(Price[i]*select[i] for i in Bids))

In [6]:
# constraints: each slot can be used at most once
# (collect the constraints in dictionary SlotAssigned for later inspection)
SlotAssigned = {}
for s in Slots:
    SlotAssigned[s] = mdl.add_constraint(mdl.sum(select[i] for i in Bids if s in SlotsInBid[i]) <= 1)

In [7]:
# constraints: at most one bid per airline
for a in Airlines:
    mdl.add_constraint(mdl.sum(select[i] for i in Bids if Airline[i]==a) <= 1)

In [8]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.0109589,status='integer optimal solution')

In [9]:
mdl.print_solution()

objective: 185
status: OPTIMAL_SOLUTION(2)
  select_4=1
  select_8=1


In [11]:
# which of the slots have been assigned in a bid?
for j in Slots:
    print("Slot " + str(j) + ": " + str(SlotAssigned[j].lhs.solution_value))

Slot 1: 1.0
Slot 2: 1.0
Slot 3: 1.0
Slot 4: 0
Slot 5: 1.0
Slot 6: 1.0


In [12]:
# is the current optimal solution unique?

# store solution as CurrentSolution
CurrentSolution = {}
for i in Bids:
    CurrentSolution[i] = select[i].solution_value

# define logical constraint to forbid current solution
# (use >=0.999 and <=0.001 because CPLEX uses floating points to represent integers)
SumOnes = mdl.sum(select[i] for i in select if CurrentSolution[i]>=0.999)
SumZeroes = mdl.sum((1-select[i]) for i in select if CurrentSolution[i]<=0.001)
mdl.add_constraint(SumOnes + SumZeroes <= 7)

docplex.mp.LinearConstraint[](-select_1-select_2-select_3+select_4-select_5-select_6-select_7+select_8+6,LE,7)

In [13]:
CurrentSolution

{1: 0, 2: 0, 3: 0, 4: 1.0, 5: 0, 6: 0, 7: 0, 8: 1.0}

In [12]:
# resolve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0,status='integer optimal solution')

In [13]:
mdl.print_solution()

objective: 170
  select_3=1
  select_5=1


In [14]:
# conclusion: because this solution has a lower obejctive value, the original solution is unique